# Experimento final: predicción de churn con priorización top-N y explicabilidad SHAP

Este notebook implementa el pipeline reproducible de extremo a extremo para:

- preparar los datos,
- construir un baseline,
- entrenar un modelo de machine learning,
- evaluar métricas globales y top-N,
- generar explicaciones SHAP,
- exportar resultados operativos en CSV y Excel.

KPI rector:
- Recall@10
- Recall@25

## Objetivo experimental

Evaluar si el modelo de aprendizaje automático supera al baseline en la priorización operativa de clientes con riesgo de churn, utilizando como criterio principal Recall@10 y Recall@25, y como criterios complementarios Lift@10/25, Precision@10/25, AUC, F1-score y SHAP.

In [1]:
import json
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

import shap
import joblib

warnings.filterwarnings("ignore")

In [2]:
RANDOM_STATE = 42
TEST_SIZE = 0.20
TOP_N_VALUES = [10, 25]
N_BOOTSTRAP = 1000

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

In [3]:
dataset_filename = "input_dataset.csv"

In [4]:
# Parameters
dataset_filename = "input_dataset.csv"


In [5]:
project_root = Path.cwd().resolve().parent

data_raw_path = project_root / "data" / "raw"
data_processed_path = project_root / "data" / "processed"
data_exports_path = project_root / "data" / "exports"

outputs_figures_path = project_root / "outputs" / "figures"
outputs_tables_path = project_root / "outputs" / "tables"
outputs_models_path = project_root / "outputs" / "models"

reports_path = project_root / "reports"

for folder in [
    data_raw_path,
    data_processed_path,
    data_exports_path,
    outputs_figures_path,
    outputs_tables_path,
    outputs_models_path,
    reports_path
]:
    folder.mkdir(parents=True, exist_ok=True)

dataset_path = data_raw_path / dataset_filename

print("Dataset filename:", dataset_filename)
print("Dataset path:", dataset_path)
print("Project root:", project_root)
print("Reports path:", reports_path)

Dataset filename: input_dataset.csv
Dataset path: C:\Users\Javier\Downloads\EF4\churn-3001e-mvp\data\raw\input_dataset.csv
Project root: C:\Users\Javier\Downloads\EF4\churn-3001e-mvp
Reports path: C:\Users\Javier\Downloads\EF4\churn-3001e-mvp\reports


## Registro metodológico inicial

Decisiones congeladas:

- Dataset: Telco Customer Churn
- Variable objetivo: Churn
- Clase positiva: Yes = 1
- Clase negativa: No = 0
- KPI rector: Recall@10 y Recall@25
- Baseline oficial: score por reglas
- Modelo principal: Random Forest
- Conjunto de prueba: ciego hasta la evaluación final
- Salida operativa mínima: top-N exportable en CSV y Excel con motivos del riesgo

In [6]:
if not dataset_path.exists():
    raise FileNotFoundError(f"No se encontró el dataset en la ruta esperada: {dataset_path}")

df = pd.read_csv(dataset_path)

if df.empty:
    raise ValueError("El dataset cargado está vacío.")

print("Archivo cargado:", dataset_path.name)
print("Dimensiones iniciales:", df.shape)
df.head()

Archivo cargado: input_dataset.csv
Dimensiones iniciales: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [8]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customerID,7043,7043,3186-AJIEK,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,7043,2,Male,3555,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SeniorCitizen,7043.0,NaN,NaN,NaN,0.162147,0.368612,0.0,0.0,0.0,0.0,1.0
Partner,7043,2,No,3641,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dependents,7043,2,No,4933,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure,7043.0,NaN,NaN,NaN,32.371149,24.559481,0.0,9.0,29.0,55.0,72.0
PhoneService,7043,2,Yes,6361,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MultipleLines,7043,3,No,3390,NaN,NaN,NaN,NaN,NaN,NaN,NaN
InternetService,7043,3,Fiber optic,3096,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OnlineSecurity,7043,3,No,3498,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
nulls = df.isnull().sum().sort_values(ascending=False)
nulls[nulls > 0]

Series([], dtype: int64)

In [10]:
nulls_df = df.isnull().sum().reset_index()
nulls_df.columns = ["variable", "nulos"]
nulls_df = nulls_df.sort_values("nulos", ascending=False)

nulls_df.to_csv(outputs_tables_path / "tabla_nulos.csv", index=False)
nulls_df[nulls_df["nulos"] > 0]

,variable,nulos


In [11]:
blank_like_summary = pd.DataFrame({
    "variable": df.columns,
    "blancos_o_vacios": [
        (df[col].astype(str).str.strip() == "").sum() if df[col].dtype == "object" else 0
        for col in df.columns
    ]
}).sort_values("blancos_o_vacios", ascending=False)

blank_like_summary.to_csv(outputs_tables_path / "tabla_blancos_texto.csv", index=False)
blank_like_summary[blank_like_summary["blancos_o_vacios"] > 0]

,variable,blancos_o_vacios
19,TotalCharges,11


In [12]:
duplicated_count = df.duplicated().sum()
print("Duplicados:", duplicated_count)

Duplicados: 0


In [13]:
df_clean = df.copy()

for col in df_clean.select_dtypes(include="object").columns:
    df_clean[col] = df_clean[col].str.strip()

if "TotalCharges" in df_clean.columns:
    totalcharges_blanks_before = (df_clean["TotalCharges"].astype(str).str.strip() == "").sum()
    print("Valores vacíos tipo string en TotalCharges antes de coerción:", totalcharges_blanks_before)

    df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")

df_clean = df_clean.drop_duplicates().reset_index(drop=True)

quality_summary = pd.DataFrame([
    {"indicador": "filas_iniciales", "valor": int(df.shape[0])},
    {"indicador": "columnas_iniciales", "valor": int(df.shape[1])},
    {"indicador": "duplicados_crudos", "valor": int(df.duplicated().sum())},
    {"indicador": "nulos_TotalCharges_post_coercion", "valor": int(df_clean["TotalCharges"].isna().sum())}
])

quality_summary.to_csv(outputs_tables_path / "resumen_calidad_datos.csv", index=False)

print("Dimensiones después de limpieza básica:", df_clean.shape)
print("Nulos en TotalCharges después de coerción:", df_clean["TotalCharges"].isna().sum())

df_clean.to_csv(data_processed_path / "dataset_limpio.csv", index=False)
print("Archivo guardado:", data_processed_path / "dataset_limpio.csv")

Valores vacíos tipo string en TotalCharges antes de coerción: 11
Dimensiones después de limpieza básica: (7043, 21)
Nulos en TotalCharges después de coerción: 11


Archivo guardado: C:\Users\Javier\Downloads\EF4\churn-3001e-mvp\data\processed\dataset_limpio.csv


In [14]:
df_clean["Churn"] = df_clean["Churn"].map({"Yes": 1, "No": 0})

print(df_clean["Churn"].value_counts(dropna=False))
print("Nulos en target:", df_clean["Churn"].isna().sum())

Churn
0    5174
1    1869
Name: count, dtype: int64


Nulos en target: 0


In [15]:
target_distribution = df_clean["Churn"].value_counts(normalize=True).sort_index() * 100
print(target_distribution)

ax = target_distribution.plot(kind="bar")
ax.set_title("Distribución de la variable objetivo Churn")
ax.set_xlabel("Clase")
ax.set_ylabel("Porcentaje")
plt.xticks([0, 1], ["No churn (0)", "Churn (1)"], rotation=0)
plt.tight_layout()
plt.savefig(outputs_figures_path / "distribucion_target.png", dpi=300, bbox_inches="tight")
plt.show()

Churn
0    73.463013
1    26.536987
Name: proportion, dtype: float64


In [16]:
variable_dictionary = pd.DataFrame({
    "variable": df_clean.columns,
    "tipo": [str(df_clean[col].dtype) for col in df_clean.columns]
})

variable_dictionary.to_csv(data_processed_path / "diccionario_variables.csv", index=False)
variable_dictionary.head()

,variable,tipo
0,customerID,object
1,gender,object
2,SeniorCitizen,int64
3,Partner,object
4,Dependents,object


## Control de fuga de información (leakage)

En esta sección se documentan las variables excluidas o revisadas por posible riesgo de fuga de información. El objetivo es garantizar que ninguna característica incorpore información posterior al momento de predicción.

En el dataset Telco Customer Churn, la principal variable no predictora y potencialmente no útil para generalización es `customerID`, por lo que se excluye del modelado y se conserva solo como identificador operativo para exportaciones.

In [17]:
candidate_drop_columns = []

if "customerID" in df_clean.columns:
    candidate_drop_columns.append("customerID")
    customer_id_series = df_clean["customerID"].copy()
else:
    customer_id_series = pd.Series(
        [f"cliente_{i}" for i in range(len(df_clean))],
        name="customerID"
    )

df_modelado = df_clean.drop(columns=candidate_drop_columns).copy()
df_modelado.to_csv(data_processed_path / "dataset_modelado.csv", index=False)

X = df_clean.drop(columns=["Churn"] + candidate_drop_columns)
y = df_clean["Churn"].copy()

print("Variables excluidas del modelado:", candidate_drop_columns)
print("Variables predictoras:", X.shape[1])
print("Target:", y.shape)
print("Archivo guardado:", data_processed_path / "dataset_modelado.csv")

Variables excluidas del modelado: ['customerID']
Variables predictoras: 19
Target: (7043,)
Archivo guardado: C:\Users\Javier\Downloads\EF4\churn-3001e-mvp\data\processed\dataset_modelado.csv


In [18]:
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()
numerical_columns = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Variables categóricas:", len(categorical_columns))
print("Variables numéricas:", len(numerical_columns))
print("\nCategóricas:", categorical_columns)
print("\nNuméricas:", numerical_columns)

Variables categóricas: 15
Variables numéricas: 4

Categóricas: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Numéricas: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


In [19]:
X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X,
    y,
    customer_id_series,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (5634, 19) (5634,)
Test: (1409, 19) (1409,)


In [20]:
transformers = []

if numerical_columns:
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ])
    transformers.append(("num", numeric_transformer, numerical_columns))

if categorical_columns:
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    transformers.append(("cat", categorical_transformer, categorical_columns))

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop"
)

preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


In [21]:
def recall_at_n(y_true, y_scores, n):
    n = min(n, len(y_true))
    df_eval = pd.DataFrame({"y_true": y_true, "score": y_scores})
    df_eval = df_eval.sort_values("score", ascending=False).head(n)
    total_positives = np.sum(y_true)
    if total_positives == 0:
        return 0.0
    return df_eval["y_true"].sum() / total_positives

def precision_at_n(y_true, y_scores, n):
    n = min(n, len(y_true))
    df_eval = pd.DataFrame({"y_true": y_true, "score": y_scores})
    df_eval = df_eval.sort_values("score", ascending=False).head(n)
    if len(df_eval) == 0:
        return 0.0
    return df_eval["y_true"].mean()

def lift_at_n(y_true, y_scores, n):
    base_rate = np.mean(y_true)
    if base_rate == 0:
        return 0.0
    return precision_at_n(y_true, y_scores, n) / base_rate

In [22]:
def evaluate_topn_metrics(y_true, y_scores, threshold=None):
    if threshold is None:
        threshold = 0.5

    y_pred = (np.array(y_scores) >= threshold).astype(int)

    metrics = {
        "AUC": roc_auc_score(y_true, y_scores),
        "F1-score": f1_score(y_true, y_pred)
    }

    for top_n in TOP_N_VALUES:
        metrics[f"Recall@{top_n}"] = recall_at_n(np.array(y_true), np.array(y_scores), top_n)
        metrics[f"Precision@{top_n}"] = precision_at_n(np.array(y_true), np.array(y_scores), top_n)
        metrics[f"Lift@{top_n}"] = lift_at_n(np.array(y_true), np.array(y_scores), top_n)

    return metrics

def relative_improvement(model_value, baseline_value):
    if baseline_value == 0:
        return np.nan
    return ((model_value - baseline_value) / baseline_value) * 100

def run_cross_validation_for_model(X_data, y_data, pipeline, n_splits=3):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    fold_rows = []

    for fold_id, (train_idx, valid_idx) in enumerate(skf.split(X_data, y_data), start=1):
        X_tr = X_data.iloc[train_idx].copy()
        X_val = X_data.iloc[valid_idx].copy()
        y_tr = y_data.iloc[train_idx].copy()
        y_val = y_data.iloc[valid_idx].copy()

        model_fold = clone(pipeline)
        model_fold.fit(X_tr, y_tr)
        y_val_proba = model_fold.predict_proba(X_val)[:, 1]
        y_val_pred = model_fold.predict(X_val)

        row = {
            "fold": fold_id,
            "AUC": roc_auc_score(y_val, y_val_proba),
            "F1-score": f1_score(y_val, y_val_pred)
        }

        for top_n in TOP_N_VALUES:
            row[f"Recall@{top_n}"] = recall_at_n(y_val.values, y_val_proba, top_n)
            row[f"Precision@{top_n}"] = precision_at_n(y_val.values, y_val_proba, top_n)
            row[f"Lift@{top_n}"] = lift_at_n(y_val.values, y_val_proba, top_n)

        fold_rows.append(row)

    cv_results_df = pd.DataFrame(fold_rows)

    cv_summary_df = pd.DataFrame([
        {
            "metrica": col,
            "media": cv_results_df[col].mean(),
            "desviacion_std": cv_results_df[col].std()
        }
        for col in cv_results_df.columns if col != "fold"
    ])

    return cv_results_df, cv_summary_df

In [23]:
def fit_baseline_reference(dataframe):
    reference = {}

    if "MonthlyCharges" in dataframe.columns:
        reference["monthlycharges_median"] = dataframe["MonthlyCharges"].median()

    return reference

def baseline_rule_score(dataframe, reference_values):
    score = np.zeros(len(dataframe), dtype=float)

    if "tenure" in dataframe.columns:
        score += np.where(dataframe["tenure"] < 12, 1.0, 0.0)

    if "MonthlyCharges" in dataframe.columns:
        monthly_ref = reference_values.get("monthlycharges_median", dataframe["MonthlyCharges"].median())
        score += np.where(dataframe["MonthlyCharges"] > monthly_ref, 1.0, 0.0)

    if "Contract" in dataframe.columns:
        score += np.where(
            dataframe["Contract"].astype(str).str.contains("Month-to-month", case=False, na=False),
            1.5,
            0.0
        )

    if "InternetService" in dataframe.columns:
        score += np.where(
            dataframe["InternetService"].astype(str).str.contains("Fiber optic", case=False, na=False),
            1.0,
            0.0
        )

    if "PaymentMethod" in dataframe.columns:
        score += np.where(
            dataframe["PaymentMethod"].astype(str).str.contains("Electronic check", case=False, na=False),
            1.0,
            0.0
        )

    if "PaperlessBilling" in dataframe.columns:
        score += np.where(
            dataframe["PaperlessBilling"].astype(str).str.contains("Yes", case=False, na=False),
            0.5,
            0.0
        )

    return score

baseline_reference = fit_baseline_reference(X_train)

baseline_scores_train = baseline_rule_score(X_train, baseline_reference)
baseline_scores_test = baseline_rule_score(X_test, baseline_reference)

baseline_threshold = np.median(baseline_scores_train)
baseline_pred_test = (baseline_scores_test >= baseline_threshold).astype(int)

print("Threshold baseline:", baseline_threshold)

Threshold baseline: 2.5


In [24]:
baseline_metrics = {
    "AUC": roc_auc_score(y_test, baseline_scores_test) if len(np.unique(y_test)) > 1 else np.nan,
    "F1-score": f1_score(y_test, baseline_pred_test)
}

for top_n in TOP_N_VALUES:
    baseline_metrics[f"Recall@{top_n}"] = recall_at_n(y_test.values, baseline_scores_test, top_n)
    baseline_metrics[f"Precision@{top_n}"] = precision_at_n(y_test.values, baseline_scores_test, top_n)
    baseline_metrics[f"Lift@{top_n}"] = lift_at_n(y_test.values, baseline_scores_test, top_n)

baseline_metrics_df = pd.DataFrame([baseline_metrics])
baseline_metrics_df

,AUC,F1-score,Recall@10,Precision@10,Lift@10,Recall@25,Precision@25,Lift@25
0,0.806411,0.558025,0.013369,0.5,1.88369,0.042781,0.64,2.411123


In [25]:
random_forest_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        class_weight="balanced"
    ))
])

random_forest_pipeline

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [26]:
cv_folds_df, cv_summary_df = run_cross_validation_for_model(
    X_train,
    y_train,
    random_forest_pipeline,
    n_splits=3
)

cv_folds_df.to_csv(outputs_tables_path / "cv_resultados_por_fold.csv", index=False)
cv_summary_df.to_csv(outputs_tables_path / "cv_resumen_metricas.csv", index=False)

print("Resultados por fold:")
display(cv_folds_df)

print("Resumen CV:")
display(cv_summary_df)

Resultados por fold:


,fold,AUC,F1-score,Recall@10,Precision@10,Lift@10,Recall@25,Precision@25,Lift@25
0,1,0.831376,0.608779,0.016064,0.8,3.016867,0.042169,0.84,3.167711
1,2,0.834429,0.607143,0.016064,0.8,3.016867,0.044177,0.88,3.318554
2,3,0.846775,0.627219,0.018036,0.9,3.387174,0.038076,0.76,2.860281


Resumen CV:


,metrica,media,desviacion_std
0,AUC,0.837526,0.008154
1,F1-score,0.614380,0.011149
2,Recall@10,0.016722,0.001138
3,Precision@10,0.833333,0.057735
4,Lift@10,3.140303,0.213797
5,Recall@25,0.041474,0.003109
6,Precision@25,0.826667,0.061101
7,Lift@25,3.115515,0.233553


In [27]:
start_time = time.time()

random_forest_pipeline.fit(X_train, y_train)

end_time = time.time()
execution_time_seconds = end_time - start_time

print(f"Tiempo de entrenamiento: {execution_time_seconds:.2f} segundos")

Tiempo de entrenamiento: 3.22 segundos


In [28]:
y_pred = random_forest_pipeline.predict(X_test)
y_proba = random_forest_pipeline.predict_proba(X_test)[:, 1]

In [29]:
model_metrics = {
    "AUC": roc_auc_score(y_test, y_proba),
    "F1-score": f1_score(y_test, y_pred)
}

for top_n in TOP_N_VALUES:
    model_metrics[f"Recall@{top_n}"] = recall_at_n(y_test.values, y_proba, top_n)
    model_metrics[f"Precision@{top_n}"] = precision_at_n(y_test.values, y_proba, top_n)
    model_metrics[f"Lift@{top_n}"] = lift_at_n(y_test.values, y_proba, top_n)

model_metrics["Tiempo_ejecucion_segundos"] = execution_time_seconds

model_metrics_df = pd.DataFrame([model_metrics])
model_metrics_df

,AUC,F1-score,Recall@10,Precision@10,Lift@10,Recall@25,Precision@25,Lift@25,Tiempo_ejecucion_segundos
0,0.83417,0.60826,0.018717,0.7,2.637166,0.05615,0.84,3.164599,3.219699


In [30]:
train_proba = random_forest_pipeline.predict_proba(X_train)[:, 1]

risk_thresholds = {
    "q33_train": float(np.quantile(train_proba, 0.33)),
    "q66_train": float(np.quantile(train_proba, 0.66))
}

pd.DataFrame([risk_thresholds]).to_csv(outputs_tables_path / "umbrales_nivel_riesgo.csv", index=False)
risk_thresholds

{'q33_train': 0.09431206793862604, 'q66_train': 0.453783683018483}

In [31]:
def plot_and_save_confusion_matrix(cm, title, filename):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, interpolation="nearest")
    ax.set_title(title)
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Valor real")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["0", "1"])
    ax.set_yticklabels(["0", "1"])

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center")

    fig.tight_layout()
    fig.savefig(outputs_figures_path / filename, dpi=300, bbox_inches="tight")
    plt.close(fig)

cm_baseline = confusion_matrix(y_test, baseline_pred_test)
cm_model = confusion_matrix(y_test, y_pred)

print("Matriz de confusión - Baseline")
print(pd.DataFrame(cm_baseline, index=["Real 0", "Real 1"], columns=["Pred 0", "Pred 1"]))

print("\nMatriz de confusión - Modelo")
print(pd.DataFrame(cm_model, index=["Real 0", "Real 1"], columns=["Pred 0", "Pred 1"]))

plot_and_save_confusion_matrix(
    cm_baseline,
    "Matriz de confusión - Baseline",
    "matriz_confusion_baseline.png"
)

plot_and_save_confusion_matrix(
    cm_model,
    "Matriz de confusión - Modelo Random Forest",
    "matriz_confusion_modelo.png"
)

Matriz de confusión - Baseline
        Pred 0  Pred 1
Real 0     533     502
Real 1      35     339

Matriz de confusión - Modelo
        Pred 0  Pred 1
Real 0     853     182
Real 1     131     243


In [32]:
comparison_df = pd.concat(
    [
        baseline_metrics_df.assign(Modelo="Baseline reglas"),
        model_metrics_df.assign(Modelo="Random Forest")
    ],
    ignore_index=True
)

comparison_df = comparison_df[["Modelo"] + [col for col in comparison_df.columns if col != "Modelo"]]
comparison_df

,Modelo,AUC,F1-score,Recall@10,Precision@10,Lift@10,Recall@25,Precision@25,Lift@25,Tiempo_ejecucion_segundos
0,Baseline reglas,0.806411,0.558025,0.013369,0.5,1.883690,0.042781,0.64,2.411123,NaN
1,Random Forest,0.834170,0.608260,0.018717,0.7,2.637166,0.056150,0.84,3.164599,3.219699


In [33]:
improvement_rows = []

for metric_name in [
    "Recall@10", "Recall@25",
    "Precision@10", "Precision@25",
    "Lift@10", "Lift@25",
    "AUC", "F1-score"
]:
    baseline_value = baseline_metrics.get(metric_name)
    model_value = model_metrics.get(metric_name)

    improvement_rows.append({
        "metrica": metric_name,
        "baseline": baseline_value,
        "modelo": model_value,
        "mejora_relativa_pct": relative_improvement(model_value, baseline_value)
    })

improvement_df = pd.DataFrame(improvement_rows)
improvement_df.to_csv(outputs_tables_path / "mejora_relativa_baseline_vs_modelo.csv", index=False)
improvement_df

,metrica,baseline,modelo,mejora_relativa_pct
0,Recall@10,0.013369,0.018717,40.000000
1,Recall@25,0.042781,0.056150,31.250000
2,Precision@10,0.500000,0.700000,40.000000
3,Precision@25,0.640000,0.840000,31.250000
4,Lift@10,1.883690,2.637166,40.000000
5,Lift@25,2.411123,3.164599,31.250000
6,AUC,0.806411,0.834170,3.442377
7,F1-score,0.558025,0.608260,9.002403


In [34]:
def bootstrap_metric(metric_function, y_true, y_scores, n_resamples=1000, random_state=42):
    rng = np.random.default_rng(random_state)
    metrics = []
    y_true = np.array(y_true)
    y_scores = np.array(y_scores)

    for _ in range(n_resamples):
        indices = rng.integers(0, len(y_true), len(y_true))
        sample_y_true = y_true[indices]
        sample_y_scores = y_scores[indices]
        metrics.append(metric_function(sample_y_true, sample_y_scores))

    lower = np.percentile(metrics, 2.5)
    upper = np.percentile(metrics, 97.5)
    mean_value = np.mean(metrics)
    return mean_value, lower, upper

In [35]:
interval_rows = []

for top_n in TOP_N_VALUES:
    recall_mean, recall_low, recall_high = bootstrap_metric(
        lambda yt, ys: recall_at_n(yt, ys, top_n),
        y_test.values,
        y_proba,
        n_resamples=N_BOOTSTRAP,
        random_state=RANDOM_STATE
    )

    interval_rows.append({
        "metrica": f"Recall@{top_n}",
        "media": recall_mean,
        "ic95_inf": recall_low,
        "ic95_sup": recall_high
    })

intervals_df = pd.DataFrame(interval_rows)
intervals_df

,metrica,media,ic95_inf,ic95_sup
0,Recall@10,0.020528,0.012887,0.027439
1,Recall@25,0.057414,0.047119,0.067044


In [36]:
fitted_preprocessor = random_forest_pipeline.named_steps["preprocessor"]
model_rf = random_forest_pipeline.named_steps["model"]

X_test_transformed = fitted_preprocessor.transform(X_test)

if hasattr(X_test_transformed, "toarray"):
    X_test_transformed_dense = X_test_transformed.toarray()
else:
    X_test_transformed_dense = X_test_transformed

all_feature_names = fitted_preprocessor.get_feature_names_out().tolist()

print("Total de features transformadas:", len(all_feature_names))
print("Forma transformada de X_test:", X_test_transformed_dense.shape)

Total de features transformadas: 45
Forma transformada de X_test: (1409, 45)


In [37]:
variable_base_map = {
    "tenure": "Antigüedad",
    "MonthlyCharges": "Cargo mensual",
    "TotalCharges": "Cargo acumulado",
    "SeniorCitizen": "Adulto mayor",
    "gender": "Género",
    "Partner": "Tiene pareja",
    "Dependents": "Tiene dependientes",
    "PhoneService": "Servicio telefónico",
    "MultipleLines": "Líneas múltiples",
    "InternetService": "Servicio de internet",
    "OnlineSecurity": "Seguridad en línea",
    "OnlineBackup": "Respaldo en línea",
    "DeviceProtection": "Protección del dispositivo",
    "TechSupport": "Soporte técnico",
    "StreamingTV": "Streaming TV",
    "StreamingMovies": "Streaming de películas",
    "Contract": "Tipo de contrato",
    "PaperlessBilling": "Facturación sin papel",
    "PaymentMethod": "Método de pago"
}

category_value_map = {
    "Male": "masculino",
    "Female": "femenino",
    "Yes": "sí",
    "No": "no",
    "DSL": "DSL",
    "Fiber optic": "fibra óptica",
    "No internet service": "sin servicio de internet",
    "No phone service": "sin servicio telefónico",
    "Month-to-month": "mes a mes",
    "One year": "un año",
    "Two year": "dos años",
    "Electronic check": "cheque electrónico",
    "Mailed check": "cheque por correo",
    "Bank transfer (automatic)": "transferencia bancaria automática",
    "Credit card (automatic)": "tarjeta de crédito automática"
}

def translate_feature_name(feature_name):
    if feature_name.startswith("num__"):
        raw_name = feature_name.replace("num__", "")
        return variable_base_map.get(raw_name, raw_name)

    if feature_name.startswith("cat__"):
        raw_name = feature_name.replace("cat__", "")

        best_prefix = None
        for candidate in sorted(variable_base_map.keys(), key=len, reverse=True):
            prefix = candidate + "_"
            if raw_name.startswith(prefix):
                best_prefix = candidate
                break

        if best_prefix is not None:
            raw_value = raw_name[len(best_prefix) + 1:]
            translated_variable = variable_base_map.get(best_prefix, best_prefix)
            translated_value = category_value_map.get(raw_value, raw_value)
            return f"{translated_variable}: {translated_value}"

        return raw_name

    return feature_name

all_feature_names_es = [translate_feature_name(feature) for feature in all_feature_names]

translated_features_df = pd.DataFrame({
    "feature_original": all_feature_names,
    "feature_es": all_feature_names_es
})

translated_features_df.to_csv(outputs_tables_path / "diccionario_features_shap_es.csv", index=False)
translated_features_df.head(20)

,feature_original,feature_es
0,num__SeniorCitizen,Adulto mayor
1,num__tenure,Antigüedad
2,num__MonthlyCharges,Cargo mensual
3,num__TotalCharges,Cargo acumulado
4,cat__gender_Female,Género: femenino
5,cat__gender_Male,Género: masculino
6,cat__Partner_No,Tiene pareja: no
7,cat__Partner_Yes,Tiene pareja: sí
8,cat__Dependents_No,Tiene dependientes: no
9,cat__Dependents_Yes,Tiene dependientes: sí


In [38]:
### 
plt.close("all")

explainer = shap.TreeExplainer(model_rf)
shap_values_raw = explainer.shap_values(X_test_transformed_dense)

if isinstance(shap_values_raw, list):
    shap_values_positive = shap_values_raw[1] if len(shap_values_raw) > 1 else shap_values_raw[0]
else:
    shap_values_array = np.array(shap_values_raw)
    if shap_values_array.ndim == 3:
        shap_values_positive = shap_values_array[:, :, 1]
    else:
        shap_values_positive = shap_values_array

plt.figure(figsize=(10, 7))

shap.summary_plot(
    shap_values_positive,
    X_test_transformed_dense,
    feature_names=all_feature_names_es,
    show=False
)

plt.title("SHAP Summary Plot")
plt.tight_layout()
plt.savefig(outputs_figures_path / "shap_summary.png", dpi=300, bbox_inches="tight")
plt.close()

print("Archivo generado:", outputs_figures_path / "shap_summary.png")

Archivo generado: C:\Users\Javier\Downloads\EF4\churn-3001e-mvp\outputs\figures\shap_summary.png


In [39]:
plt.close("all")
plt.figure(figsize=(10, 7))

shap.summary_plot(
    shap_values_positive,
    X_test_transformed_dense,
    feature_names=all_feature_names_es,
    plot_type="bar",
    show=False
)

plt.title("SHAP Bar Plot")
plt.tight_layout()
plt.savefig(outputs_figures_path / "shap_bar.png", dpi=300, bbox_inches="tight")
plt.close()

print("Archivo generado:", outputs_figures_path / "shap_bar.png")

Archivo generado: C:\Users\Javier\Downloads\EF4\churn-3001e-mvp\outputs\figures\shap_bar.png


In [40]:
results_test = X_test.copy().reset_index(drop=True)
results_test["cliente_id"] = ids_test.reset_index(drop=True)
results_test["y_true"] = y_test.reset_index(drop=True)
results_test["probabilidad"] = y_proba
results_test["test_row_idx"] = np.arange(len(results_test))

q33 = risk_thresholds["q33_train"]
q66 = risk_thresholds["q66_train"]

def assign_risk_level(prob):
    if prob >= q66:
        return "ALTO"
    elif prob >= q33:
        return "MEDIO"
    else:
        return "BAJO"

results_test["nivel_riesgo"] = results_test["probabilidad"].apply(assign_risk_level)

results_test = results_test.sort_values("probabilidad", ascending=False).reset_index(drop=True)
results_test.head(15)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,cliente_id,y_true,probabilidad,test_row_idx,nivel_riesgo
0,Male,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.55,69.55,1820-TQVEV,1,0.972629,341,ALTO
1,Female,1,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.60,69.60,8375-DKEBR,1,0.970982,1289,ALTO
2,Female,1,No,No,1,Yes,Yes,Fiber optic,No,No,No,No,No,Yes,Month-to-month,Yes,Electronic check,85.05,85.05,1069-XAIEM,1,0.970477,1109,ALTO
3,Male,1,No,No,1,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,76.45,76.45,9248-OJYKK,1,0.968761,618,ALTO
4,Male,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.90,69.90,9804-ICWBG,1,0.967573,1252,ALTO
5,Male,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.90,69.90,5542-TBBWB,0,0.967573,629,ALTO
6,Female,1,No,No,2,Yes,Yes,Fiber optic,No,No,No,No,No,Yes,Month-to-month,Yes,Electronic check,84.05,186.05,2545-EBUPK,0,0.965862,889,ALTO
7,Male,0,No,No,1,Yes,Yes,Fiber optic,No,No,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,95.45,95.45,0295-PPHDO,1,0.962434,1221,ALTO
8,Male,0,No,No,1,No,No phone service,DSL,No,No,No,No,No,Yes,Month-to-month,Yes,Electronic check,35.55,35.55,0841-NULXI,1,0.959464,1178,ALTO
9,Male,0,No,No,7,Yes,Yes,Fiber optic,No,No,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,94.10,701.30,8161-QYMTT,0,0.957788,995,ALTO


In [41]:
top10_df = results_test.head(10).copy()
top25_df = results_test.head(25).copy()

print("Top 10:", top10_df.shape)
print("Top 25:", top25_df.shape)

Top 10: (10, 24)
Top 25: (25, 24)


In [42]:
def get_top_shap_reasons(shap_row, feature_names_es, top_k=3):
    abs_values = np.abs(shap_row)
    top_indices = np.argsort(abs_values)[::-1][:top_k]
    return [feature_names_es[i] for i in top_indices]

def assign_top_reasons(df_top):
    motivos = []
    motivo_principal = []

    for row_idx in df_top["test_row_idx"]:
        top_features = get_top_shap_reasons(
            shap_values_positive[row_idx],
            all_feature_names_es,
            top_k=3
        )
        motivos.append(" | ".join(top_features))
        motivo_principal.append(top_features[0] if len(top_features) > 0 else None)

    df_top["top_3_motivos"] = motivos
    df_top["motivo_principal"] = motivo_principal
    return df_top

top10_df = assign_top_reasons(top10_df)
top25_df = assign_top_reasons(top25_df)

top10_df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,cliente_id,y_true,probabilidad,test_row_idx,nivel_riesgo,top_3_motivos,motivo_principal
0,Male,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.55,69.55,1820-TQVEV,1,0.972629,341,ALTO,Antigüedad | Cargo acumulado | Tipo de contrat...,Antigüedad
1,Female,1,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.60,69.60,8375-DKEBR,1,0.970982,1289,ALTO,Antigüedad | Cargo acumulado | Tipo de contrat...,Antigüedad
2,Female,1,No,No,1,Yes,Yes,Fiber optic,No,No,No,No,No,Yes,Month-to-month,Yes,Electronic check,85.05,85.05,1069-XAIEM,1,0.970477,1109,ALTO,Antigüedad | Tipo de contrato: mes a mes | Car...,Antigüedad
3,Male,1,No,No,1,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,76.45,76.45,9248-OJYKK,1,0.968761,618,ALTO,Antigüedad | Tipo de contrato: mes a mes | Car...,Antigüedad
4,Male,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.90,69.90,9804-ICWBG,1,0.967573,1252,ALTO,Antigüedad | Cargo acumulado | Tipo de contrat...,Antigüedad


In [43]:
results_test = assign_top_reasons(results_test.copy())

cols_for_web = [
    col for col in [
        "cliente_id",
        "probabilidad",
        "nivel_riesgo",
        "motivo_principal",
        "top_3_motivos",
        "Contract",
        "InternetService",
        "PaymentMethod",
        "tenure",
        "MonthlyCharges",
        "TotalCharges",
        "SeniorCitizen",
        "Partner",
        "Dependents"
    ] if col in results_test.columns
]

full_scored_df = results_test[cols_for_web].copy()
full_scored_df.to_csv(data_exports_path / "clientes_scoreados_completos.csv", index=False)

print("Archivo web completo guardado en:", data_exports_path / "clientes_scoreados_completos.csv")
full_scored_df.head()

Archivo web completo guardado en: C:\Users\Javier\Downloads\EF4\churn-3001e-mvp\data\exports\clientes_scoreados_completos.csv


,cliente_id,probabilidad,nivel_riesgo,motivo_principal,top_3_motivos,Contract,InternetService,PaymentMethod,tenure,MonthlyCharges,TotalCharges,SeniorCitizen,Partner,Dependents
0,1820-TQVEV,0.972629,ALTO,Antigüedad,Antigüedad | Cargo acumulado | Tipo de contrat...,Month-to-month,Fiber optic,Electronic check,1,69.55,69.55,0,No,No
1,8375-DKEBR,0.970982,ALTO,Antigüedad,Antigüedad | Cargo acumulado | Tipo de contrat...,Month-to-month,Fiber optic,Electronic check,1,69.60,69.60,1,No,No
2,1069-XAIEM,0.970477,ALTO,Antigüedad,Antigüedad | Tipo de contrato: mes a mes | Car...,Month-to-month,Fiber optic,Electronic check,1,85.05,85.05,1,No,No
3,9248-OJYKK,0.968761,ALTO,Antigüedad,Antigüedad | Tipo de contrato: mes a mes | Car...,Month-to-month,Fiber optic,Electronic check,1,76.45,76.45,1,No,No
4,9804-ICWBG,0.967573,ALTO,Antigüedad,Antigüedad | Cargo acumulado | Tipo de contrat...,Month-to-month,Fiber optic,Electronic check,1,69.90,69.90,0,No,No


In [44]:
top10_df.to_csv(data_exports_path / "top10_clientes_riesgo.csv", index=False)
top25_df.to_csv(data_exports_path / "top25_clientes_riesgo.csv", index=False)

print("Archivos CSV exportados correctamente.")

Archivos CSV exportados correctamente.


In [45]:
top10_df.to_excel(data_exports_path / "top10_clientes_riesgo.xlsx", index=False)
top25_df.to_excel(data_exports_path / "top25_clientes_riesgo.xlsx", index=False)

print("Archivos Excel exportados correctamente.")

Archivos Excel exportados correctamente.


In [46]:
baseline_metrics_df.to_csv(outputs_tables_path / "tabla_metricas_baseline.csv", index=False)
model_metrics_df.to_csv(outputs_tables_path / "tabla_metricas_modelo.csv", index=False)
comparison_df.to_csv(outputs_tables_path / "comparativa_baseline_vs_modelo.csv", index=False)
intervals_df.to_csv(outputs_tables_path / "intervalos_confianza.csv", index=False)
improvement_df.to_csv(outputs_tables_path / "mejora_relativa_baseline_vs_modelo.csv", index=False)

improv_recall_10 = relative_improvement(model_metrics["Recall@10"], baseline_metrics["Recall@10"])
improv_recall_25 = relative_improvement(model_metrics["Recall@25"], baseline_metrics["Recall@25"])

precision_not_worse = (
    model_metrics["Precision@10"] >= baseline_metrics["Precision@10"] and
    model_metrics["Precision@25"] >= baseline_metrics["Precision@25"]
)

checklist_df = pd.DataFrame([
    {"criterio": "Se generó dataset limpio", "cumple": "Sí" if (data_processed_path / "dataset_limpio.csv").exists() else "No"},
    {"criterio": "Se generó dataset modelado", "cumple": "Sí" if (data_processed_path / "dataset_modelado.csv").exists() else "No"},
    {"criterio": "Se documentó calidad de datos", "cumple": "Sí" if (outputs_tables_path / "resumen_calidad_datos.csv").exists() else "No"},
    {"criterio": "Se calculó Recall@10", "cumple": "Sí" if "Recall@10" in model_metrics else "No"},
    {"criterio": "Se calculó Recall@25", "cumple": "Sí" if "Recall@25" in model_metrics else "No"},
    {"criterio": "Se ejecutó validación cruzada", "cumple": "Sí" if (outputs_tables_path / "cv_resumen_metricas.csv").exists() else "No"},
    {"criterio": "Se generó explicación SHAP global", "cumple": "Sí" if (outputs_figures_path / "shap_summary.png").exists() else "No"},
    {"criterio": "Se generó ranking top-10 exportable", "cumple": "Sí" if (data_exports_path / "top10_clientes_riesgo.csv").exists() else "No"},
    {"criterio": "Se generó ranking top-25 exportable", "cumple": "Sí" if (data_exports_path / "top25_clientes_riesgo.csv").exists() else "No"},
    {"criterio": "El modelo mejora >=10% en Recall@10", "cumple": "Sí" if improv_recall_10 >= 10 else "No"},
    {"criterio": "El modelo mejora >=10% en Recall@25", "cumple": "Sí" if improv_recall_25 >= 10 else "No"},
    {"criterio": "La precisión no empeora en top-N", "cumple": "Sí" if precision_not_worse else "No"}
])

checklist_df.to_csv(outputs_tables_path / "checklist_validacion_mvp.csv", index=False)
checklist_df

,criterio,cumple
0,Se generó dataset limpio,Sí
1,Se generó dataset modelado,Sí
2,Se documentó calidad de datos,Sí
3,Se calculó Recall@10,Sí
4,Se calculó Recall@25,Sí
5,Se ejecutó validación cruzada,Sí
6,Se generó explicación SHAP global,Sí
7,Se generó ranking top-10 exportable,Sí
8,Se generó ranking top-25 exportable,Sí
9,El modelo mejora >=10% en Recall@10,Sí


In [47]:
joblib.dump(random_forest_pipeline, outputs_models_path / "modelo_random_forest.pkl")
joblib.dump(fitted_preprocessor, outputs_models_path / "preprocessor_pipeline.pkl")

print("Modelo guardado en:", outputs_models_path / "modelo_random_forest.pkl")
print("Preprocesador guardado en:", outputs_models_path / "preprocessor_pipeline.pkl")

Modelo guardado en: C:\Users\Javier\Downloads\EF4\churn-3001e-mvp\outputs\models\modelo_random_forest.pkl
Preprocesador guardado en: C:\Users\Javier\Downloads\EF4\churn-3001e-mvp\outputs\models\preprocessor_pipeline.pkl


In [48]:
execution_log_df = pd.DataFrame([
    {"artefacto": "dataset_limpio.csv", "ruta": str(data_processed_path / "dataset_limpio.csv"), "existe": (data_processed_path / "dataset_limpio.csv").exists()},
    {"artefacto": "dataset_modelado.csv", "ruta": str(data_processed_path / "dataset_modelado.csv"), "existe": (data_processed_path / "dataset_modelado.csv").exists()},
    {"artefacto": "diccionario_variables.csv", "ruta": str(data_processed_path / "diccionario_variables.csv"), "existe": (data_processed_path / "diccionario_variables.csv").exists()},
    {"artefacto": "top10_clientes_riesgo.csv", "ruta": str(data_exports_path / "top10_clientes_riesgo.csv"), "existe": (data_exports_path / "top10_clientes_riesgo.csv").exists()},
    {"artefacto": "top25_clientes_riesgo.csv", "ruta": str(data_exports_path / "top25_clientes_riesgo.csv"), "existe": (data_exports_path / "top25_clientes_riesgo.csv").exists()},
    {"artefacto": "top10_clientes_riesgo.xlsx", "ruta": str(data_exports_path / "top10_clientes_riesgo.xlsx"), "existe": (data_exports_path / "top10_clientes_riesgo.xlsx").exists()},
    {"artefacto": "top25_clientes_riesgo.xlsx", "ruta": str(data_exports_path / "top25_clientes_riesgo.xlsx"), "existe": (data_exports_path / "top25_clientes_riesgo.xlsx").exists()},
    {"artefacto": "shap_summary.png", "ruta": str(outputs_figures_path / "shap_summary.png"), "existe": (outputs_figures_path / "shap_summary.png").exists()},
    {"artefacto": "shap_bar.png", "ruta": str(outputs_figures_path / "shap_bar.png"), "existe": (outputs_figures_path / "shap_bar.png").exists()},
    {"artefacto": "modelo_random_forest.pkl", "ruta": str(outputs_models_path / "modelo_random_forest.pkl"), "existe": (outputs_models_path / "modelo_random_forest.pkl").exists()},
    {"artefacto": "preprocessor_pipeline.pkl", "ruta": str(outputs_models_path / "preprocessor_pipeline.pkl"), "existe": (outputs_models_path / "preprocessor_pipeline.pkl").exists()}
])

execution_log_df.to_csv(outputs_tables_path / "bitacora_ejecucion_notebook.csv", index=False)
execution_log_df

,artefacto,ruta,existe
0,dataset_limpio.csv,C:\Users\Javier\Downloads\EF4\churn-3001e-mvp\...,True
1,dataset_modelado.csv,C:\Users\Javier\Downloads\EF4\churn-3001e-mvp\...,True
2,diccionario_variables.csv,C:\Users\Javier\Downloads\EF4\churn-3001e-mvp\...,True
3,top10_clientes_riesgo.csv,C:\Users\Javier\Downloads\EF4\churn-3001e-mvp\...,True
4,top25_clientes_riesgo.csv,C:\Users\Javier\Downloads\EF4\churn-3001e-mvp\...,True
5,top10_clientes_riesgo.xlsx,C:\Users\Javier\Downloads\EF4\churn-3001e-mvp\...,True
6,top25_clientes_riesgo.xlsx,C:\Users\Javier\Downloads\EF4\churn-3001e-mvp\...,True
7,shap_summary.png,C:\Users\Javier\Downloads\EF4\churn-3001e-mvp\...,True
8,shap_bar.png,C:\Users\Javier\Downloads\EF4\churn-3001e-mvp\...,True
9,modelo_random_forest.pkl,C:\Users\Javier\Downloads\EF4\churn-3001e-mvp\...,True


In [49]:
def to_serializable(obj):
    if isinstance(obj, dict):
        return {k: to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [to_serializable(x) for x in obj]
    elif isinstance(obj, tuple):
        return [to_serializable(x) for x in obj]
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.bool_):
        return bool(obj)
    else:
        return obj

improv_recall_10 = relative_improvement(model_metrics["Recall@10"], baseline_metrics["Recall@10"])
improv_recall_25 = relative_improvement(model_metrics["Recall@25"], baseline_metrics["Recall@25"])
improv_lift_10 = relative_improvement(model_metrics["Lift@10"], baseline_metrics["Lift@10"])
improv_lift_25 = relative_improvement(model_metrics["Lift@25"], baseline_metrics["Lift@25"])

precision_not_worse = (
    model_metrics["Precision@10"] >= baseline_metrics["Precision@10"] and
    model_metrics["Precision@25"] >= baseline_metrics["Precision@25"]
)

cumple_mvp = bool(
    (improv_recall_10 >= 10) and
    (improv_recall_25 >= 10) and
    (precision_not_worse)
)

final_summary = {
    "kpi_rector": {
        "Recall@10": model_metrics.get("Recall@10"),
        "Recall@25": model_metrics.get("Recall@25")
    },
    "baseline": baseline_metrics,
    "modelo_ml": model_metrics,
    "mejora_relativa_pct": {
        "Recall@10": improv_recall_10,
        "Recall@25": improv_recall_25,
        "Lift@10": improv_lift_10,
        "Lift@25": improv_lift_25
    },
    "precision_no_empeora": precision_not_worse,
    "cumple_mvp": cumple_mvp
}

final_summary_serializable = to_serializable(final_summary)

print(json.dumps(final_summary_serializable, indent=4, ensure_ascii=False))

summary_md = f"""
# Resumen de resultados

## KPI rector
- Recall@10: {final_summary_serializable["kpi_rector"]["Recall@10"]}
- Recall@25: {final_summary_serializable["kpi_rector"]["Recall@25"]}

## Mejora relativa
- Recall@10: {final_summary_serializable["mejora_relativa_pct"]["Recall@10"]:.2f} %
- Recall@25: {final_summary_serializable["mejora_relativa_pct"]["Recall@25"]:.2f} %
- Lift@10: {final_summary_serializable["mejora_relativa_pct"]["Lift@10"]:.2f} %
- Lift@25: {final_summary_serializable["mejora_relativa_pct"]["Lift@25"]:.2f} %

## Precisión no empeora
- {"Sí" if final_summary_serializable["precision_no_empeora"] else "No"}

## Veredicto preliminar del MVP
- Cumple MVP: {"Sí" if final_summary_serializable["cumple_mvp"] else "No"}
"""

with open(reports_path / "resumen_resultados.md", "w", encoding="utf-8") as f:
    f.write(summary_md)

print("Resumen guardado en:", reports_path / "resumen_resultados.md")

{
    "kpi_rector": {
        "Recall@10": 0.01871657754010695,
        "Recall@25": 0.05614973262032086
    },
    "baseline": {
        "AUC": 0.8064106538531091,
        "F1-score": 0.5580246913580247,
        "Recall@10": 0.013368983957219251,
        "Precision@10": 0.5,
        "Lift@10": 1.8836898395721924,
        "Recall@25": 0.0427807486631016,
        "Precision@25": 0.64,
        "Lift@25": 2.4111229946524064
    },
    "modelo_ml": {
        "AUC": 0.8341703479810897,
        "F1-score": 0.6082603254067585,
        "Recall@10": 0.01871657754010695,
        "Precision@10": 0.7,
        "Lift@10": 2.637165775401069,
        "Recall@25": 0.05614973262032086,
        "Precision@25": 0.84,
        "Lift@25": 3.164598930481283,
        "Tiempo_ejecucion_segundos": 3.219698667526245
    },
    "mejora_relativa_pct": {
        "Recall@10": 39.99999999999999,
        "Recall@25": 31.25000000000001,
        "Lift@10": 39.999999999999986,
        "Lift@25": 31.24999999999998
    },
 

## Conclusión técnica preliminar

Con base en las métricas obtenidas, se determinará si el modelo de aprendizaje automático supera al baseline en el KPI rector (Recall@10 y Recall@25), y si el MVP cumple con los criterios definidos para la validación experimental del proyecto.

Este notebook deja evidencia reproducible de:
- preparación de datos,
- control básico de leakage,
- comparación baseline vs modelo,
- evaluación top-N,
- explicabilidad global y local con SHAP,
- exportación operativa de resultados,
- guardado de artefactos finales del MVP.